In [1]:
import pandas as pd
import numpy as np

df = pd.read_pickle("../data/processed/movies_features.pkl")
df.shape

(1719, 20)

In [2]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   # remove punctuation and numbers
    text = re.sub(r"\s+", " ", text).strip()  # collapse multiple spaces
    return text

df["overview_clean"] = df["overview"].apply(clean_text)
df[["overview", "overview_clean"]].head(3)

,overview,overview_clean
0,On a perilous climb across Thailand's Mount Kw...,on a perilous climb across thailand s mount kw...
1,Two sisters move to the country with their fat...,two sisters move to the country with their fat...
2,"A passionate telling of the story of Sada Abe,...",a passionate telling of the story of sada abe ...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english",
)

tfidf_matrix = vectorizer.fit_transform(df["overview_clean"])
tfidf_matrix.shape

(1719, 5000)

In [18]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
avg_scores = tfidf_matrix.mean(axis=0).A1  # average TF-IDF score per word across all movies

top_20_idx = avg_scores.argsort()[-20:][::-1]
for i in top_20_idx:
    print(feature_names[i], round(avg_scores[i], 4))

world 0.0179
life 0.0165
new 0.0164
family 0.0149
young 0.0133
man 0.0121
finds 0.0096
home 0.0092
mysterious 0.0088
war 0.0086
old 0.0084
love 0.0083
group 0.0082
time 0.0082
story 0.0081
help 0.0081
years 0.0079
lives 0.0078
woman 0.0076
friends 0.0076


In [19]:
movie_idx = 0
row = tfidf_matrix[movie_idx].toarray().flatten()
top_words_idx = row.argsort()[-10:][::-1]

print("Title:", df.iloc[movie_idx]["title"])
for i in top_words_idx:
    if row[i] > 0:
        print(" -", feature_names[i], round(row[i], 3))

Title: Fall 2: Deadpoint
 - impossible odds 0.271
 - inducing heights 0.271
 - inducing 0.271
 - climb 0.26
 - heights 0.252
 - ground 0.252
 - thailand 0.245
 - mount 0.24
 - thousands 0.235
 - moment 0.235


In [20]:
import joblib
from pathlib import Path
from scipy import sparse

Path("../models").mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
sparse.save_npz("../models/tfidf_matrix.npz", tfidf_matrix)

df.to_pickle("../data/processed/movies_features.pkl")  # includes overview_clean now
print("saved")

saved
